# AI/ML Commercialization Decision Engine

**Flow:** Customer signals → ML pattern detection → AI explanation → Commercial decision

This notebook presents the ranked portfolio, model outputs, SHAP explainability, and executive summary.

> **Interactive dashboard:** run `streamlit run app/streamlit_app.py` from the project root.

## 1. Setup & Run Pipeline

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from models.decision_engine import run_decision_engine
from models.insight_layer import build_insights, generate_executive_summary, FEATURE_LABELS

PROCESSED = ROOT / "data" / "processed"
FEATURES_PATH = PROCESSED / "concept_features.csv"

if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Run Phase 1–2 first:\n"
        "  python data/generate_mock_data.py\n"
        "  python data/prepare_features.py"
    )

report, artifacts = run_decision_engine(FEATURES_PATH, PROCESSED)
insights = build_insights(artifacts)
ranked = insights.sort_values("portfolio_rank")

print(f"Concepts analyzed: {len(ranked)}")
print(f"Top concept: {ranked.iloc[0]['concept_name']} ({ranked.iloc[0]['readiness_score']}/100)")

## 2. Executive Summary

In [ ]:
summary = generate_executive_summary(artifacts["ranked"])
print(summary)

## 3. Ranked Portfolio Recommendations

In [ ]:
display_cols = [
    "portfolio_rank", "concept_name", "industry", "readiness_score",
    "confidence_score", "recommended_outcome", "cluster_profile", "ai_narrative",
]
ranked[display_cols]

## 4. Visual Analytics

In [ ]:
OUTCOME_COLORS = {
    "MVP Build": "#2ecc71",
    "Customer Pilot": "#3498db",
    "Reusable Asset": "#9b59b6",
    "Incubate": "#f39c12",
    "Archive": "#e74c3c",
}

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Readiness ranking
ordered = ranked.sort_values("readiness_score", ascending=True)
colors = [OUTCOME_COLORS.get(o, "#95a5a6") for o in ordered["recommended_outcome"]]
axes[0, 0].barh(ordered["concept_name"], ordered["readiness_score"], color=colors)
axes[0, 0].set_xlabel("Readiness Score (1–100)")
axes[0, 0].set_title("Concept Readiness Ranking")
axes[0, 0].set_xlim(0, 100)

# Outcome distribution
counts = ranked["recommended_outcome"].value_counts()
pie_colors = [OUTCOME_COLORS.get(o, "#95a5a6") for o in counts.index]
axes[0, 1].pie(counts, labels=counts.index, autopct="%1.0f%%", colors=pie_colors, startangle=140)
axes[0, 1].set_title("Recommended Outcome Distribution")

# K-Means clusters
df = artifacts["df"]
palette = sns.color_palette("Set2", df["cluster_id"].nunique())
for cid, group in df.groupby("cluster_id"):
    axes[1, 0].scatter(
        group["demand_intensity"], group["feasibility_risk"],
        s=group["readiness_score"] * 4, alpha=0.75,
        label=group["cluster_profile"].iloc[0], color=palette[cid % len(palette)],
    )
axes[1, 0].set_xlabel("Demand Intensity")
axes[1, 0].set_ylabel("Feasibility Risk")
axes[1, 0].set_title("K-Means Clusters: Demand vs Effort")
axes[1, 0].legend(fontsize=7, loc="best")

# Feature importance
importance = artifacts["importance"].head(8)
sns.barplot(data=importance, x="importance", y="feature", palette="Blues_r", ax=axes[1, 1])
axes[1, 1].set_title("Random Forest Feature Importance")

plt.tight_layout()
plt.show()

## 5. SHAP Explainability — Why Each Decision?

In [ ]:
feature_names = artifacts["feature_columns"]

for _, row in ranked.iterrows():
    shap_cols = [f"shap_{f}" for f in feature_names if f"shap_{f}" in row.index]
    shap_vals = [(c.replace("shap_", ""), row[c]) for c in shap_cols]
    shap_vals.sort(key=lambda x: abs(x[1]), reverse=True)
    top = shap_vals[:4]

    print(f"\n{'='*70}")
    print(f"#{int(row['portfolio_rank'])} {row['concept_name']} → {row['recommended_outcome']}")
    print(f"Readiness: {row['readiness_score']}/100 | Confidence: {row['confidence_score']:.0%}")
    print(f"\nAI Narrative: {row['ai_narrative']}")
    print("\nTop SHAP drivers:")
    for feat, val in top:
        label = FEATURE_LABELS.get(feat, feat)
        direction = "↑ toward outcome" if val > 0 else "↓ against outcome"
        print(f"  • {label}: {val:+.4f} ({direction})")

## 6. SHAP Waterfall — Selected Concept

In [ ]:
CONCEPT = ranked.iloc[0]["concept_name"]  # change to explore others
row = ranked[ranked["concept_name"] == CONCEPT].iloc[0]

shap_cols = [f"shap_{f}" for f in feature_names if f"shap_{f}" in row.index]
labels = [FEATURE_LABELS.get(c.replace("shap_", ""), c) for c in shap_cols]
values = [row[c] for c in shap_cols]
pairs = sorted(zip(labels, values), key=lambda x: abs(x[1]), reverse=True)[:8]
labels, values = zip(*pairs)
bar_colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(labels, values, color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP contribution toward predicted outcome")
ax.set_title(f"Why: {row['concept_name']} → {row['recommended_outcome']}")
plt.tight_layout()
plt.show()

## 7. Cluster Summary

In [ ]:
artifacts["cluster_summary"]

## 8. Model Report

In [ ]:
with open(PROCESSED / "model_report.json") as f:
    model_report = json.load(f)

print("Baseline weights:", model_report["baseline_model"]["weights"])
print("\nOutcome distribution:", model_report["classifier"]["outcome_distribution"])
print("\nTop features:")
for feat in model_report["top_features"]:
    print(f"  {feat['feature']}: {feat['importance']:.3f}")

## 9. Streamlit Dashboard

For interactive exploration (concept drill-down, live filters), launch the Streamlit app from the project root:

```bash
streamlit run app/streamlit_app.py
```